# 07 – Ligand–receptor links for run B (early lymphoid → stroma)

**Question.** Which L–R links go into the reverse CausalCCC run?

Run B uses **the same cells** as run A (notebook 04: at most 500 per population; the export in
`results/thymus_early/04_runA_input/` is reused), with the roles swapped: senders = early lymphoid cells,
receivers = stromal cells. Only the L–R links change. They are chosen with the same rules as in notebook 04:

- the paper's claims that pass the expression filter in this direction (notebook 03) are always included:
  LTB → LTBR, RANKL → RANK (TNFSF11 → TNFRSF11A), CD40LG → CD40, and the two Notch pairs that also pass in this
  direction (JAG1 → NOTCH1, DLL1 → NOTCH1);
- the remaining links are ranked by the S5 mean (oriented, notebook 03), up to 25 S5 pairs;
- a pair is skipped if all its gene links are already in the list; pairs whose "receptor" is IDE are excluded.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
RES = ROOT / "results/thymus_early"
OUT = RES / "07_runB_links"
OUT.mkdir(parents=True, exist_ok=True)

N_LINKS = 25
MAX_GOI = 30          # causalCCC.MIselection(): at most 30 genes of interest per side

DB = THY / "cellphonedb_v2_db/db_2019-09"
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")

def genes_of(name):
    if name in cplx.index:
        units = cplx.loc[name, ["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()
        return [gene.gene_name[u] for u in units]
    return [name]

In [2]:
cand = pd.read_csv(RES / "03_ligand_receptor_candidates/candidates_B_lymphoid_to_stroma.csv")
CLAIMED = ["LTB_LTBR", "TNFSF11_TNFRSF11A", "CD40LG_CD40", "JAG1_NOTCH1", "DLL1_NOTCH1"]
EXCLUDED_RECEPTORS = {"IDE"}
assert set(CLAIMED) <= set(cand.pair)

cand["claimed"] = cand.pair.isin(CLAIMED)
ranked = cand.sort_values(["claimed", "s5_max_mean"], ascending=[False, False]).reset_index(drop=True)

chosen, skipped, gene_links = [], [], set()
for _, row in ranked.iterrows():
    if len(chosen) == N_LINKS:
        break
    new_links = {(l, r) for l in genes_of(row.ligand) for r in genes_of(row.receptor)}
    if set(genes_of(row.receptor)) & EXCLUDED_RECEPTORS:
        skipped.append((row.pair, "receptor is an enzyme (IDE)"))
    elif new_links <= gene_links:
        skipped.append((row.pair, "all gene links already included"))
    else:
        chosen.append(row.pair)
        gene_links |= new_links

links = ranked[ranked.pair.isin(chosen)].copy()
cols = ["pair", "ligand", "receptor", "orientation_rule", "claimed", "s5_max_mean", "s5_best_pair",
        "max_ligand_frac", "ligand_pops", "max_receptor_frac", "receptor_pops"]
print("skipped while filling the list:", skipped)
links[cols].round(2)

skipped while filling the list: [('CCL3_IDE', 'receptor is an enzyme (IDE)')]


,pair,ligand,receptor,orientation_rule,claimed,s5_max_mean,s5_best_pair,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops
0,LTB_LTBR,LTB,LTBR,receptor flag,True,2.04,ILC3 -> Fb_1,0.94,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.45,"cTEC, mTEC(II), Fb_1, Endo"
1,JAG1_NOTCH1,JAG1,NOTCH1,receptor flag,True,0.38,ILC3 -> Endo,0.28,"ETP, ILC3, NK",0.52,"cTEC, mTEC(II), Endo"
2,CD40LG_CD40,CD40LG,CD40,receptor flag,True,0.35,ILC3 -> Endo,0.13,ILC3,0.69,"cTEC, mTEC(II), Endo"
3,DLL1_NOTCH1,DLL1,NOTCH1,receptor flag,True,0.24,ILC3 -> Endo,0.35,ILC3,0.52,"cTEC, mTEC(II), Endo"
4,TNFSF11_TNFRSF11A,TNFSF11,TNFRSF11A,manual,True,0.24,ILC3 -> mTEC(II),0.26,ILC3,0.54,"cTEC, mTEC(II)"
5,CD74_MIF,MIF,CD74,receptor flag,False,2.59,DN(early) -> mTEC(II),0.97,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.97,"cTEC, mTEC(II), Fb_1, Endo"
6,ACKR1_CCL5,CCL5,ACKR1,receptor flag,False,1.99,NK -> Endo,0.36,"ILC3, NK",0.52,"mTEC(II), Endo"
7,CD99_PILRA,CD99,PILRA,receptor flag,False,1.84,DN(Q) -> mTEC(II),1.00,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.22,mTEC(II)
8,APP_CD74,APP,CD74,receptor flag,False,1.76,ETP -> mTEC(II),0.65,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT",0.97,"cTEC, mTEC(II), Fb_1, Endo"
9,PECAM1_CD38,CD38,PECAM1,ambiguous,False,1.37,NK -> Endo,0.50,"ETP, DN(early), DN(P), DN(Q), γδT",0.97,Endo


In [3]:
edges = pd.DataFrame([dict(ligands=l, receptors=r)
                      for _, row in links.iterrows()
                      for l in genes_of(row.ligand) for r in genes_of(row.receptor)]).drop_duplicates()
ligand_genes, receptor_genes = sorted(edges.ligands.unique()), sorted(edges.receptors.unique())
print(len(links), "L-R pairs ->", len(edges), "gene-gene links")
print("ligand genes:  ", len(ligand_genes), ligand_genes)
print("receptor genes:", len(receptor_genes), receptor_genes)
assert len(ligand_genes) <= MAX_GOI and len(receptor_genes) <= MAX_GOI

genes_in_matrix = set(pd.read_csv(RES / "04_runA_input/genes.csv").gene)
assert set(ligand_genes + receptor_genes) <= genes_in_matrix

edges.to_csv(OUT / "interact_edges.tsv", sep="\t", index=False)
links[cols].to_csv(OUT / "selected_links.csv", index=False)
print("written to", OUT)

25 L-R pairs -> 26 gene-gene links
ligand genes:   22 ['APP', 'AREG', 'CCL5', 'CD38', 'CD40LG', 'CD99', 'CXCL1', 'DLL1', 'FASLG', 'IFNG', 'ITGAL', 'ITGB2', 'JAG1', 'JAM3', 'KLRF1', 'LTA', 'LTB', 'MDK', 'MIF', 'SPN', 'TGFB1', 'TNFSF11']
receptor genes: 19 ['ACKR1', 'CD40', 'CD74', 'CLEC2B', 'EGFR', 'ICAM1', 'IFNGR1', 'IFNGR2', 'JAM2', 'LRP1', 'LTBR', 'NOTCH1', 'PECAM1', 'PILRA', 'TGFBR3', 'TNFRSF10D', 'TNFRSF11A', 'TNFRSF14', 'TNFRSF1A']
written to ../../results/thymus_early/07_runB_links


## Summary

| | |
|---|---|
| Cells | same as run A (notebook 04); senders = `early_lymphoid` (2,622), receivers = `stroma` (1,555) |
| L–R pairs | 25 S5 pairs = 5 claimed pairs + 20 by S5 mean → 26 gene–gene links |
| Skipped while filling | `CCL3_IDE` (IDE is an enzyme) |
| Genes of interest | 22 ligand genes, 19 receptor genes (limit 30 per side) |
| Written to | `results/thymus_early/07_runB_links/` |

As in run A, the claimed pairs have low S5 means (0.24–0.38) except LTB → LTBR (2.04, second highest overall), and the
rest of the list is dominated by broadly expressed genes (MIF, CD74, APP, CD99).